# Pairs Trading on Borsa Istanbul
## Replication of Gatev, Goetzmann & Rouwenhorst (2006)

**EC581 Algorithmic Trading — Bogazici University**

This notebook replicates the *Pairs Trading* strategy of Gatev, Goetzmann & Rouwenhorst (2006) on Turkish equities (2010–2026). After each section a comparison table maps every implementation decision back to the original paper.

Run **Kernel → Restart & Run All** to reproduce every table and figure.

## 0. Setup & Imports

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import backtrader as bt

from src.pairs import (
    normalize_prices, compute_ssd, select_top_pairs,
    compute_locked_sigma, liquidity_filter,
)
from src.backtest import (
    run_ggr_pair_backtest, simulate_pair_returns, run_ggr_portfolio,
)
from src.metrics import sharpe_ratio, max_drawdown, monte_carlo_test

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# GGR parameters — all match the paper exactly
FORMATION_DAYS = 252   # 12 months
TRADING_DAYS   = 126   # 6 months
ROLL_DAYS      = 21    # 1 month step
TOP_N          = 20    # top pairs per portfolio
ENTRY_SIGMA    = 2.0   # 2σ divergence threshold
COMMISSION_BPS = 10.0  # 10 bps per leg (realistic for BIST)

print('Setup complete.')

## 1. Data Loading

Adjusted close prices for all BIST tickers are loaded from `data/prices.csv`. Run `python data_download.py` once to fetch or refresh. The GGR liquidity filter — complete price history during the formation window — is applied dynamically at every rolling step.

In [ ]:
prices = pd.read_csv('data/prices.csv', index_col=0, parse_dates=True)
print(f'Prices: {prices.shape[1]} tickers × {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

In [ ]:
norm = prices / prices.iloc[0] * 100
fig, ax = plt.subplots(figsize=(14, 5))
norm.plot(ax=ax, alpha=0.35, legend=False, linewidth=0.7)
ax.set_title('BIST Adjusted Close Prices — Normalized to 100 (2010–2026)')
ax.set_ylabel('Index (Day 1 = 100)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/normalized_prices.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Market | NYSE / AMEX / NASDAQ (CRSP) | Borsa Istanbul (Yahoo Finance) | Different market, same method |
| Sample period | 1962–2002 (40 years) | 2010–2026 (16 years) | Shorter; fewer regime cycles |
| Universe size | ~8 274 stocks screened per window | ~70–130 stocks per window | Smaller; fewer pairs |
| Price data | Daily total-return (div. adjusted) | Daily adj. close (Yahoo Finance) | ✓ Dividend-adjusted |
| Liquidity rule | Complete return history in 12-month formation window | No NaN prices in formation window | ✓ Exact match |


## 2. Formation Period — Exhaustive SSD Pair Selection

GGR normalizes each stock's price to a cumulative-return index starting at 1.0 on the first day of the formation window:

$$P^*_{i,t} = P_{i,t} / P_{i,0}$$

The **Sum of Squared Deviations** (SSD) measures how closely two normalized series tracked each other over the 12-month window:

$$SSD_{A,B} = \sum_{t=1}^{252}\left(P^*_{A,t} - P^*_{B,t}\right)^2$$

All N(N−1)/2 pairs are ranked by SSD and the **Top 20** (smallest SSD) are selected. The spread standard deviation σ computed over the formation window is then **locked** and used unchanged as the trading threshold for the full 6-month trading period.

In [ ]:
DEMO_FORMATION_START = '2018-01-02'
DEMO_FORMATION_END   = '2018-12-31'
DEMO_TRADING_START   = '2019-01-02'
DEMO_TRADING_END     = '2019-06-28'

liquid = liquidity_filter(prices, DEMO_FORMATION_START, DEMO_FORMATION_END)
print(f'Tickers with complete prices: {len(liquid)} / {len(prices.columns)}')

norm_form = normalize_prices(prices.loc[DEMO_FORMATION_START:DEMO_FORMATION_END, liquid])
ssd_df = compute_ssd(norm_form)
print(f'Pairs computed: {len(ssd_df):,}  ({len(liquid)} tickers)')

top_pairs = select_top_pairs(ssd_df, n=TOP_N)
sigmas    = compute_locked_sigma(norm_form, top_pairs)

top_df = pd.DataFrame([
    {'rank': i+1, 'ticker1': t1.replace('.IS',''), 'ticker2': t2.replace('.IS',''),
     'SSD': round(ssd,4), 'locked_sigma': round(sigmas.get((t1,t2),0),5),
     'entry_threshold_2sigma': round(2*sigmas.get((t1,t2),0),5)}
    for i,(t1,t2,ssd) in enumerate(top_pairs)
])
top_df.to_csv(f'{RESULTS_DIR}/top20_pairs.csv', index=False)
print('\nTop 20 pairs by SSD:')
top_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for idx, (t1, t2, ssd_val) in enumerate(top_pairs[:6]):
    ax = axes[idx]
    norm_form[[t1, t2]].plot(ax=ax, linewidth=1.5)
    ax.set_title(f'{t1.replace(".IS","")} / {t2.replace(".IS","")}\n'
                 f'SSD={ssd_val:.3f}  σ={sigmas.get((t1,t2),0):.4f}', fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=8)
plt.suptitle('Top 6 Pairs — Normalized Prices in Formation Window (2018)', y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/formation_top6_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(min(40, len(ssd_df))), ssd_df['ssd'].head(40), color='steelblue', alpha=0.8)
ax.axvline(TOP_N - 0.5, color='red', linestyle='--', linewidth=1.5,
           label=f'Top-{TOP_N} cutoff')
ax.set_xlabel('Pair rank (0 = smallest SSD)')
ax.set_ylabel('SSD')
ax.set_title('SSD Distribution — Top 40 Pairs (2018 Formation Window)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ssd_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Normalization | P*_{i,t} = P_{i,t}/P_{i,0}, starts at 1.0 on day 1 of formation | Identical formula in normalize_prices() | ✓ Exact match |
| Pair metric | SSD = Σ(P*_A − P*_B)²  over 252 days | Identical formula in compute_ssd() | ✓ Exact match |
| Pair selection | Exhaustive search over all N(N−1)/2 pairs; take Top-5 and Top-20 | Exhaustive search; take Top-20 | ✓ Top-20 matches; Top-5 not reported separately |
| Formation σ | σ = std(P*_A − P*_B) over formation window; frozen for trading | Identical; passed as locked_sigma parameter | ✓ Exact match |
| Formation length | 12 months (≈252 trading days) | FORMATION_DAYS = 252 | ✓ Exact match |


## 3. Trading Engine — Backtrader Demo (Single Window)

The GGR trading rule monitors the normalized spread P\*_A − P\*_B each day against the **locked** 2σ threshold. When the spread diverges beyond 2σ we open a **dollar-neutral** position: short \$1 of the outperformer, long \$1 of the underperformer. We hold until the spread **crosses zero** (full convergence) or the 6-month trading window expires (time-stop). Commission of 10 bps per leg is deducted at entry and exit.

In [ ]:
p0_row = prices.loc[DEMO_FORMATION_START]

print(f'Trading window: {DEMO_TRADING_START} → {DEMO_TRADING_END}')
print(f'Entry threshold: 2σ (locked from formation)')
print(f'Exit: spread crosses zero')
print(f'Commission: {COMMISSION_BPS} bps per leg')
print()

bt_results = {}
for t1, t2, _ in top_pairs[:5]:
    sigma = sigmas.get((t1, t2))
    if sigma is None or sigma < 1e-10:
        continue
    p1_0 = p0_row.get(t1, np.nan)
    p2_0 = p0_row.get(t2, np.nan)
    if pd.isna(p1_0) or pd.isna(p2_0):
        continue
    tp1 = prices.loc[DEMO_TRADING_START:DEMO_TRADING_END, t1].dropna()
    tp2 = prices.loc[DEMO_TRADING_START:DEMO_TRADING_END, t2].dropna()
    common = tp1.index.intersection(tp2.index)
    res = run_ggr_pair_backtest(
        tp1.loc[common], tp2.loc[common],
        p1_0=p1_0, p2_0=p2_0, locked_sigma=sigma,
        pair_name=f'{t1}_{t2}', entry_sigma=ENTRY_SIGMA,
        commission_bps=COMMISSION_BPS,
    )
    bt_results[f'{t1}_{t2}'.replace('.IS','')] = res
    print(f'  {t1.replace(".IS","")}/{t2.replace(".IS","")}: '
          f'Sharpe={res["sharpe"]:.3f}  Return={res["total_return_pct"]:.2f}%  '
          f'MaxDD={res["max_drawdown_pct"]:.1f}%  Trades={res["n_trades"]}')

bt_df = pd.DataFrame({
    k: {'Sharpe': round(v['sharpe'],3), 'Return %': round(v['total_return_pct'],2),
        'Max DD %': round(v['max_drawdown_pct'],2), 'Trades': v['n_trades']}
    for k,v in bt_results.items()
}).T
bt_df.to_csv(f'{RESULTS_DIR}/bt_demo_results.csv')
bt_df

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, res in bt_results.items():
    ec = res['equity_curve']
    if len(ec) > 2:
        (1 + ec).cumprod().plot(ax=ax, label=name)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(f'GGR Strategy — Equity Curves (2019 Trading Window, {COMMISSION_BPS} bps)')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/bt_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Position type | Dollar-neutral: short \$1 outperformer, long \$1 underperformer | Two separate Backtrader feeds; sell/buy each leg individually | ✓ Exact match |
| Entry signal | |spread| > 2σ  where σ is the locked formation std | GGRStrategy checks spread > 2×locked_sigma each bar | ✓ Exact match |
| Exit signal | Spread crosses zero (P*_A = P*_B) | spread >= 0 or spread <= 0 triggers close() | ✓ Exact match |
| Time-stop | Force close at end of 6-month trading window | stop() closes any open position at data end | ✓ Exact match |
| Commission | 0% in Panel A; ~162 bps round-trip estimated for Panel B | 10 bps per leg = 40 bps round-trip | Conservative realistic estimate for BIST |
| Position sizing | \$1 per leg (dollar-neutral) | cash_per_leg / price = integer shares per leg | ✓ Dollar-neutral intent; integer rounding |


## 4. Walk-Forward Overlapping Portfolio Simulation (2012–2026)

GGR does not backtest one static pair. Instead it runs a **rolling machine** that launches a new Top-20 portfolio every month: the 12-month formation window advances by 1 month, selects a fresh set of pairs with freshly locked σ values, and opens a 6-month trading window. Because each trading window is 6 months long and windows roll monthly, up to **6 portfolios trade simultaneously** at any point. The strategy's daily return is the **equal-weighted average** of all active portfolios (committed-capital convention: inactive pair slots return 0).

We replicate this exactly with `run_ggr_portfolio`.

In [ ]:
print('Running GGR walk-forward portfolio simulation ...')
print(f'  Formation {FORMATION_DAYS}d | Trading {TRADING_DAYS}d | Roll {ROLL_DAYS}d | '
      f'Top-{TOP_N} | Entry {ENTRY_SIGMA}σ | Commission {COMMISSION_BPS} bps')

port = run_ggr_portfolio(
    prices=prices,
    formation_days=FORMATION_DAYS,
    trading_days=TRADING_DAYS,
    roll_days=ROLL_DAYS,
    top_n=TOP_N,
    entry_sigma=ENTRY_SIGMA,
    commission_bps=COMMISSION_BPS,
)

print(f'Windows: {port["n_windows"]}  |  '
      f'Sharpe: {port["sharpe"]:.3f}  |  '
      f'MaxDD: {port["max_drawdown"]*100:.1f}%  |  '
      f'Total Return: {port["total_return_pct"]:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ec = port['equity_curve']
ec.plot(ax=axes[0], color='steelblue', linewidth=1.5)
axes[0].axhline(1.0, color='black', linewidth=0.7, linestyle='--')
axes[0].set_title(f'GGR Portfolio — Committed Capital Return ({COMMISSION_BPS} bps commission)')
axes[0].set_ylabel('Cumulative Return')

rets = port['portfolio_returns'].replace(0, np.nan).dropna()
rets.rolling(252).apply(lambda x: x.mean()/x.std()*np.sqrt(252) if x.std()>0 else 0
).plot(ax=axes[1], color='darkorange', linewidth=1.2)
axes[1].axhline(0, color='black', linewidth=0.7, linestyle='--')
axes[1].set_title('Rolling 1-Year Sharpe Ratio')
axes[1].set_ylabel('Sharpe')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/portfolio_equity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ws_df = pd.DataFrame(port['window_stats'])
if not ws_df.empty:
    ws_df.to_csv(f'{RESULTS_DIR}/window_stats.csv', index=False)
    print('Per-window statistics (first 10):')
    print(ws_df.head(10).to_string(index=False))

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Roll frequency | New portfolio initiated every calendar month | ROLL_DAYS = 21 trading days | ✓ Equivalent |
| Max concurrent | Up to 6 portfolios active simultaneously | trading_days/roll_days = 126/21 = 6 | ✓ Exact match |
| Pair count | Top 20 per portfolio window | TOP_N = 20 | ✓ Exact match |
| Return aggregation | Equal-weighted average across all active windows | port_sum / active_count per day | ✓ Exact match |
| Capital convention | Committed capital: denominator = 20 pair slots | Pad inactive pairs with 0; take mean over 20 | ✓ Exact match |
| Backtest engine | Not specified (academic simulation) | pandas simulate_pair_returns (Backtrader for demo only) | Same logic; pandas for speed |


## 5. Performance Analysis

GGR (2006) Table 1 reports monthly excess returns for Top-20 portfolios over 1962–2002. Their benchmark is the committed-capital return (denominator = 20 pair slots regardless of how many are active). We report the same metric for BIST over 2012–2026.

In [ ]:
port_rets = port['portfolio_returns'].replace(0, np.nan).dropna()

ann_ret = float(port_rets.mean() * 252 * 100)
ann_vol = float(port_rets.std() * np.sqrt(252) * 100)
sharpe  = port['sharpe']
mdd     = port['max_drawdown'] * 100
tot_ret = port['total_return_pct']

print('=== GGR Portfolio — Committed Capital ===')
print(f'Annualised Return : {ann_ret:.2f}%')
print(f'Annualised Vol    : {ann_vol:.2f}%')
print(f'Sharpe Ratio      : {sharpe:.3f}')
print(f'Max Drawdown      : {mdd:.1f}%')
print(f'Total Return      : {tot_ret:.1f}%')

perf_df = pd.DataFrame({
    'Metric': ['Ann. Return %','Ann. Vol %','Sharpe','Max DD %','Total Return %'],
    'GGR Paper (US 1962-2002, 0 cost)': ['~9.6%','~3.5%','~N/A','N/A','N/A'],
    'This Study (BIST 2012-2026)': [f'{ann_ret:.2f}%', f'{ann_vol:.2f}%',
                                    f'{sharpe:.3f}', f'{mdd:.1f}%', f'{tot_ret:.1f}%'],
})
perf_df.to_csv(f'{RESULTS_DIR}/portfolio_performance.csv', index=False)
perf_df

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Return basis | Committed capital (÷20 pair slots) | mean over 20 padded slots → same denominator | ✓ Exact match |
| Paper benchmark | Top-20: 0.78%/month excess return (committed) | Reported above; BIST result differs | Different market/period |
| Fully-invested | Also reported: 1.31%/month (÷active pairs only) | Not separately computed in this study | Minor gap; committed is the main metric |
| Risk-free rate | Subtracted from raw returns | Set to 0 in Sharpe computation | Slight optimism; TL risk-free rate is high |
| Market exposure | Near-zero beta (self-financing long-short) | Same structure; beta not separately computed | Same logic |


## 6. Monte Carlo Significance Test

GGR §3.2 tests whether the pair returns are significantly different from a bootstrap null that accounts for momentum and reversal in individual stocks. We implement a **sign-randomization permutation test**: under the null hypothesis that the strategy has no edge, each daily return's sign is equally likely to be positive or negative. We flip signs 1 000 times, recompute the Sharpe each time, and count how often the random Sharpe exceeds the observed Sharpe. That fraction is the empirical p-value.

In [ ]:
if len(port_rets) > 20:
    mc = monte_carlo_test(port_rets, n_simulations=1000, seed=42)
    print(f'Observed Sharpe : {mc["observed_sharpe"]:.3f}')
    print(f'Null mean       : {mc["null_mean"]:.3f} ± {mc["null_std"]:.3f}')
    print(f'p-value         : {mc["p_value"]:.4f}')
    print(f'Significant     : {"YES (p<0.05)" if mc["p_value"] < 0.05 else "NO"}')

    fig, ax = plt.subplots(figsize=(10, 4))
    null_v = mc['null_sharpes']
    bins = 40 if (null_v.max() - null_v.min()) > 1e-9 else 1
    ax.hist(null_v, bins=bins, color='steelblue', alpha=0.7,
            label='Sign-randomized null')
    ax.axvline(mc['observed_sharpe'], color='red', linewidth=2,
               label=f'Observed = {mc["observed_sharpe"]:.3f}')
    ax.set_title(f'Monte Carlo Significance Test | p = {mc["p_value"]:.4f}')
    ax.set_xlabel('Sharpe Ratio')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/mc_test.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient data for Monte Carlo test.')

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Test type | Bootstrap resampling of daily stock returns (preserves cross-section) | Sign-randomization of strategy daily returns | Different null; same spirit |
| Null hypothesis | Returns are no different from momentum/reversal portfolios | Returns are symmetric around zero (no directional edge) | Weaker null; ours tests positive mean |
| Simulations | Not specified (large bootstrap) | 1 000 sign-randomizations | Sufficient for p-value precision |
| Paper result | Significant excess returns across all sub-periods | Reported above | Comparison in output |


## 7. Transaction Cost Sensitivity

GGR Table 1 Panel A reports zero-commission results; Panel B applies an estimated bid-ask correction of roughly **162 bps per round-trip** for the 1962–1988 period, which roughly halves the gross returns. For BIST, realistic costs are lower in absolute terms but still material. We compare gross (0 bps) against three realistic cost levels.

In [ ]:
cost_scenarios = [0, 5, 10, 20]
cost_results = []
for bps in cost_scenarios:
    p = run_ggr_portfolio(
        prices=prices,
        formation_days=FORMATION_DAYS, trading_days=TRADING_DAYS,
        roll_days=ROLL_DAYS, top_n=TOP_N,
        entry_sigma=ENTRY_SIGMA, commission_bps=bps,
    )
    cost_results.append({'commission_bps': bps,
                         'sharpe': round(p['sharpe'],3),
                         'total_return_pct': round(p['total_return_pct'],1),
                         'max_drawdown_pct': round(p['max_drawdown']*100,1)})
    print(f'  {bps:3d} bps: Sharpe={p["sharpe"]:.3f}  Return={p["total_return_pct"]:.1f}%')

cost_df = pd.DataFrame(cost_results)
cost_df.to_csv(f'{RESULTS_DIR}/transaction_cost_sensitivity.csv', index=False)
cost_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels = [str(r['commission_bps'])+' bps' for r in cost_results]
axes[0].bar(labels, [r['sharpe'] for r in cost_results], color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Sharpe vs Commission')
axes[0].set_ylabel('Annualised Sharpe')
axes[1].bar(labels, [r['total_return_pct'] for r in cost_results], color='coral', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Total Return % vs Commission')
axes[1].set_ylabel('Total Return %')
plt.suptitle('GGR Portfolio — Transaction Cost Sensitivity')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cost_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 📄 Paper vs. Implementation

| Dimension | GGR (2006) | This Study | Match? |
|-----------|-----------|------------|--------|
| Panel A (paper) | Zero commission — gross returns | 0 bps scenario in the table above | ✓ Matches |
| Panel B (paper) | Bid-ask correction ~162 bps round-trip (1962–1988) | 10 bps/leg = 40 bps round-trip (BIST 2012–2026) | Lower cost; BIST has narrower spreads for large-caps |
| Cost components | Bid-ask spread (no explicit brokerage modeled) | Single per-leg commission; no market impact modeled | Conservative lower bound |
| Conclusion | ~50% of gross profit survives after bid-ask (US 1962–88) | Sensitivity shown for 0/5/10/20 bps scenarios | Equivalent analysis |


## 8. Conclusions

### Overall Methodology Match

| Component | GGR (2006) | This Study | Status |
|-----------|-----------|------------|--------|
| Universe | All liquid CRSP stocks | All liquid BIST stocks | Different market |
| Liquidity filter | Complete return history | No NaN prices in formation | ✓ Exact |
| Normalization | P*=P/P₀ from formation start | Identical | ✓ Exact |
| Pair selection | Exhaustive SSD, Top-20 | Exhaustive SSD, Top-20 | ✓ Exact |
| Locked σ | Frozen at formation end | Passed as fixed parameter | ✓ Exact |
| Entry rule | \|spread\| > 2σ | \|spread\| > 2σ | ✓ Exact |
| Exit rule | Spread crosses zero | spread >= 0 / <= 0 | ✓ Exact |
| Time-stop | End of 6-month window | Backtrader stop() / last bar | ✓ Exact |
| Portfolio roll | Monthly, 6 concurrent | ROLL_DAYS=21, 6 concurrent | ✓ Exact |
| Return basis | Committed capital (÷20) | Mean over 20 padded slots | ✓ Exact |
| Commission | 0% Panel A / ~162 bps Panel B | 10 bps per leg | Realistic for BIST |

### Would We Invest?
A real deployment requires all three:
1. **Positive post-cost Sharpe** — survives at realistic commission levels
2. **Statistical significance** — Monte Carlo p < 0.05
3. **Regime robustness** — rolling Sharpe stays positive across sub-periods

BIST-specific risks not in the original paper: TL currency volatility, capital controls, regulatory interventions, and thin liquidity for smaller names that dominate the SSD ranking.